# 03 - Modelo Baseline de Credit Scoring
## Regressão Logística

**Tech Challenge - Fase 04 | Machine Learning Engineering**

Este notebook treina e avalia o modelo baseline do projeto de **Credit Scoring**.

A Regressão Logística foi escolhida como baseline por ser adequada para classificação binária, rápida, interpretável e amplamente utilizada como referência em problemas de risco de crédito.

### Objetivos

- reutilizar as regras de feature engineering já validadas;
- trabalhar com um carregamento otimizado para a memória da máquina local;
- manter o conjunto de teste completo;
- treinar o baseline em uma amostra estratificada grande do conjunto de treino;
- avaliar o modelo com métricas adequadas para classes desbalanceadas;
- inspecionar erros, probabilidades e coeficientes;
- salvar o pipeline completo para reutilização posterior.

> **Nota computacional:** o dataset possui mais de 1,3 milhão de empréstimos na população final de modelagem. O conjunto de teste permanece completo, enquanto o treinamento do baseline utiliza uma amostra estratificada de 300 mil registros para evitar estouros de memória no ambiente local. Isso não altera o objetivo do baseline, que é estabelecer uma referência inicial de desempenho.

# 0. Configuração do ambiente

In [ ]:
from pathlib import Path
import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from credit_scoring_observability.preprocessing import (
    BASELINE_FEATURES,
    GOOD_STATUS,
    BAD_STATUS,
    engineer_features,
    split_features_target,
    split_train_test,
)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

COLORS = {
    "blue": "#377DFF",
    "teal": "#00A6A6",
    "green": "#2CB67D",
    "gold": "#F4B942",
    "coral": "#FF6B6B",
    "purple": "#7B61FF",
    "navy": "#17324D",
    "gray": "#6B7280",
}

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# 1. Caminhos e parâmetros

In [ ]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
MODEL_PATH = Path("../models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_PATH / "credit_scoring_baseline.joblib"
METRICS_FILE = MODEL_PATH / "baseline_metrics.json"

CHUNK_SIZE = 25_000
TRAIN_SAMPLE_SIZE = 300_000
RANDOM_STATE = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATA_PATH.resolve()}\n"
        "Coloque accepted_2007_to_2018Q4.csv em data/raw/."
    )

print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Modelo será salvo em: {MODEL_FILE.resolve()}")

# 2. Carregamento otimizado para memória

O carregamento é feito apenas com as 23 features aprovadas no EDA, além de `loan_status` e `issue_d`.

Para reduzir o consumo de memória:

- o arquivo é lido em chunks de 25 mil linhas;
- apenas empréstimos com desfecho conhecido são mantidos;
- colunas `float64` são convertidas para `float32`;
- strings repetitivas de baixa cardinalidade são convertidas para `category`.

`term` e `emp_length` permanecem como texto porque ainda serão transformadas por `engineer_features()`.

In [ ]:
cols_to_load = BASELINE_FEATURES + [
    "loan_status",
    "issue_d",
]

frames = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=cols_to_load,
    chunksize=CHUNK_SIZE,
    low_memory=True,
):
    filtered = chunk[
        chunk["loan_status"].isin(
            GOOD_STATUS + BAD_STATUS
        )
    ].copy()

    float_cols = filtered.select_dtypes(
        include=["float64"]
    ).columns

    filtered[float_cols] = filtered[
        float_cols
    ].astype("float32")

    frames.append(filtered)

df = pd.concat(
    frames,
    ignore_index=True,
)

del frames
gc.collect()

category_cols = [
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
    "loan_status",
]

for col in category_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

df["target"] = (
    df["loan_status"]
    .isin(BAD_STATUS)
    .astype("int8")
)

memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Shape carregado: {df.shape}")
print(f"Memória utilizada: {memory_mb:,.2f} MB")

A população esperada possui aproximadamente **1.348.099 empréstimos** com desfecho conhecido.

# 3. Feature engineering

In [ ]:
df["term"] = df["term"].astype("object")
df["emp_length"] = df["emp_length"].astype("object")

df = engineer_features(df)

X, y, metadata = split_features_target(df)

print(f"Features: {X.shape}")
print(f"Target:   {y.shape}")
print(f"Taxa de inadimplência: {y.mean():.2%}")

# 4. Train/Test Split

O conjunto de teste corresponde a 20% da população e permanece **integralmente preservado** para avaliação final.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Treino completo: {X_train.shape}")
print(f"Teste completo:  {X_test.shape}")

split_distribution = pd.DataFrame({
    "Treino": y_train.value_counts(normalize=True).sort_index(),
    "Teste": y_test.value_counts(normalize=True).sort_index(),
}).T

split_distribution.columns = [
    "Não inadimplente",
    "Inadimplente",
]

split_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(split_distribution.index))
width = 0.32

ax.bar(
    x - width/2,
    split_distribution["Não inadimplente"] * 100,
    width,
    label="Não inadimplente",
    color=COLORS["teal"],
)

ax.bar(
    x + width/2,
    split_distribution["Inadimplente"] * 100,
    width,
    label="Inadimplente",
    color=COLORS["coral"],
)

ax.set_xticks(x)
ax.set_xticklabels(split_distribution.index)
ax.set_ylabel("Percentual")
ax.set_title("Distribuição do target após o split estratificado")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

# 5. Amostra de treinamento do baseline

O conjunto de treino original possui mais de 1 milhão de registros. Para o baseline, será utilizada uma amostra estratificada de até 300 mil registros.

O teste continua completo.

In [ ]:
train_size = min(TRAIN_SAMPLE_SIZE, len(X_train))

X_train_model, _, y_train_model, _ = train_test_split(
    X_train,
    y_train,
    train_size=train_size,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

print(f"Treino original:  {X_train.shape}")
print(f"Treino baseline:  {X_train_model.shape}")
print(f"Teste completo:   {X_test.shape}")

print("\nDistribuição do treino baseline:")
print(y_train_model.value_counts(normalize=True))

# 6. Otimização dos tipos numéricos

In [ ]:
numeric_features = (
    X_train_model
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_features = (
    X_train_model
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

for col in numeric_features:
    X_train_model[col] = X_train_model[col].astype("float32")
    X_test[col] = X_test[col].astype("float32")

for col in categorical_features:
    X_train_model[col] = X_train_model[col].astype("object")
    X_test[col] = X_test[col].astype("object")

print(f"Numéricas: {len(numeric_features)}")
print(numeric_features)

print(f"\nCategóricas: {len(categorical_features)}")
print(categorical_features)

## Liberação de memória

In [ ]:
del X_train
del y_train
del df
del X
del y
del metadata

gc.collect()

print("Objetos intermediários removidos da memória.")

# 7. Pipeline de pré-processamento do baseline

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                copy=False,
            ),
        ),
        (
            "scaler",
            RobustScaler(
                copy=False,
            ),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    sparse_threshold=1.0,
)

# 8. Modelo baseline

A Regressão Logística utiliza o solver `saga`, adequado para conjuntos grandes e matrizes esparsas. O threshold inicial permanece em **0,50**.

In [ ]:
baseline_model = LogisticRegression(
    solver="saga",
    max_iter=300,
    tol=1e-3,
    random_state=RANDOM_STATE,
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", baseline_model),
    ]
)

baseline_pipeline

# 9. Treinamento

In [ ]:
start = time.perf_counter()

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")

    baseline_pipeline.fit(
        X_train_model,
        y_train_model,
    )

elapsed = time.perf_counter() - start

print("Modelo treinado com sucesso.")
print(f"Tempo de treinamento: {elapsed / 60:.2f} minutos")

if caught_warnings:
    print("\nWarnings durante o treinamento:")
    for warning in caught_warnings:
        print(f"- {warning.message}")
else:
    print("\nNenhum warning durante o treinamento.")

# 10. Predições no conjunto de teste completo

In [ ]:
start_pred = time.perf_counter()

y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

prediction_elapsed = time.perf_counter() - start_pred

print(f"Predições realizadas em {prediction_elapsed:.2f} segundos.")
print(f"Quantidade de registros avaliados: {len(y_test):,}")

# 11. Métricas do baseline

Como o target é aproximadamente 80/20, a acurácia isolada pode ser enganosa. Por isso serão consideradas Accuracy, Precision, Recall, F1, ROC-AUC e PR-AUC.

In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "pr_auc": average_precision_score(y_test, y_proba),
}

metrics_df = pd.Series(metrics, name="score").to_frame()
metrics_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))

plot_metrics = metrics_df.loc[
    ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
]

bars = ax.bar(
    plot_metrics.index,
    plot_metrics["score"] * 100,
    color=[
        COLORS["blue"],
        COLORS["teal"],
        COLORS["coral"],
        COLORS["purple"],
        COLORS["green"],
        COLORS["gold"],
    ],
)

ax.set_title("Métricas do modelo baseline")
ax.set_ylabel("Score")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_ylim(0, 105)
ax.tick_params(axis="x", rotation=20)

for bar, value in zip(bars, plot_metrics["score"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1.5,
        f"{value:.3f}",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

# 12. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Não inadimplente",
            "Inadimplente",
        ],
        digits=4,
    )
)

# 13. Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

confusion_summary = pd.DataFrame(
    {"Quantidade": [tn, fp, fn, tp]},
    index=[
        "Verdadeiro negativo",
        "Falso positivo",
        "Falso negativo",
        "Verdadeiro positivo",
    ],
)

confusion_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Não inadimplente",
        "Inadimplente",
    ],
).plot(
    cmap="Blues",
    values_format=",",
    ax=ax,
    colorbar=False,
)

ax.set_title("Matriz de Confusão - Baseline")

plt.tight_layout()
plt.show()

### Interpretação de negócio

- **Falso negativo:** cliente inadimplente classificado como não inadimplente. Pode representar exposição a perda de crédito.
- **Falso positivo:** cliente adimplente classificado como inadimplente. Pode representar recusa desnecessária ou perda de oportunidade.

# 14. Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_predictions(
    y_test,
    y_proba,
    name="Regressão Logística",
    color=COLORS["purple"],
    ax=ax,
)

ax.set_title("Curva ROC - Modelo Baseline")

plt.tight_layout()
plt.show()

# 15. Curva Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

PrecisionRecallDisplay.from_predictions(
    y_test,
    y_proba,
    name="Regressão Logística",
    color=COLORS["coral"],
    ax=ax,
)

ax.set_title("Curva Precision-Recall - Modelo Baseline")

plt.tight_layout()
plt.show()

# 16. Distribuição das probabilidades previstas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    y_proba[y_test.values == 0],
    bins=50,
    alpha=0.62,
    density=True,
    color=COLORS["teal"],
    label="Não inadimplente",
)

ax.hist(
    y_proba[y_test.values == 1],
    bins=50,
    alpha=0.60,
    density=True,
    color=COLORS["coral"],
    label="Inadimplente",
)

ax.axvline(
    0.50,
    linestyle="--",
    linewidth=2,
    color=COLORS["navy"],
    label="Threshold = 0,50",
)

ax.set_title("Distribuição da probabilidade prevista de inadimplência")
ax.set_xlabel("Probabilidade prevista")
ax.set_ylabel("Densidade")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

# 17. Sensibilidade ao threshold

In [ ]:
threshold_rows = []

for threshold in np.arange(0.20, 0.81, 0.10):
    pred_t = (y_proba >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    threshold_df["threshold"],
    threshold_df["precision"],
    marker="o",
    color=COLORS["blue"],
    label="Precision",
)

ax.plot(
    threshold_df["threshold"],
    threshold_df["recall"],
    marker="o",
    color=COLORS["coral"],
    label="Recall",
)

ax.plot(
    threshold_df["threshold"],
    threshold_df["f1"],
    marker="o",
    color=COLORS["purple"],
    label="F1",
)

ax.set_title("Sensibilidade das métricas ao threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

# 18. Interpretabilidade da Regressão Logística

Os coeficientes indicam associação no modelo e não devem ser interpretados como causalidade.

In [ ]:
fitted_preprocessor = baseline_pipeline.named_steps["preprocessor"]
fitted_model = baseline_pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_model.coef_[0]

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
    })
    .sort_values("coefficient")
)

coef_df.head()

In [ ]:
top_negative = coef_df.head(12)
top_positive = coef_df.tail(12)

fig, ax = plt.subplots(figsize=(10, 7))

combined = pd.concat(
    [top_negative, top_positive]
).sort_values("coefficient")

colors = [
    COLORS["teal"] if value < 0 else COLORS["coral"]
    for value in combined["coefficient"]
]

ax.barh(
    combined["feature"],
    combined["coefficient"],
    color=colors,
)

ax.axvline(0, color=COLORS["navy"], linewidth=1)
ax.set_title("Principais coeficientes da Regressão Logística")
ax.set_xlabel("Coeficiente")

plt.tight_layout()
plt.show()

# 19. Salvamento do pipeline completo

In [ ]:
joblib.dump(
    baseline_pipeline,
    MODEL_FILE,
)

with METRICS_FILE.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {key: float(value) for key, value in metrics.items()},
        file,
        indent=2,
    )

print(f"Modelo salvo:    {MODEL_FILE}")
print(f"Métricas salvas: {METRICS_FILE}")
print(f"Modelo existe:   {MODEL_FILE.exists()}")
print(f"Métricas existem:{METRICS_FILE.exists()}")

# 20. Teste de recarga do modelo

In [ ]:
reloaded_model = joblib.load(MODEL_FILE)

reload_sample = X_test.head(5)
original_prob = baseline_pipeline.predict_proba(reload_sample)[:, 1]
reloaded_prob = reloaded_model.predict_proba(reload_sample)[:, 1]

reload_check = np.allclose(
    original_prob,
    reloaded_prob,
)

print(f"Predições idênticas após recarga: {reload_check}")

pd.DataFrame({
    "prob_original": original_prob,
    "prob_recarregada": reloaded_prob,
})

In [ ]:
if not reload_check:
    raise AssertionError(
        "As probabilidades após recarregar o modelo não são idênticas."
    )

print("Pipeline salvo e recarregado com sucesso.")

# 21. Conclusões do baseline

O modelo baseline foi construído com uma Regressão Logística integrada ao pipeline completo de pré-processamento.

### Estratégia computacional
- dataset de modelagem: aproximadamente 1,35 milhão de registros;
- treino original: aproximadamente 1,08 milhão;
- treino do baseline: amostra estratificada de até 300 mil;
- teste final: conjunto completo com aproximadamente 270 mil registros.

### Avaliação
As métricas de maior interesse são **ROC-AUC, PR-AUC, Recall e F1**, pois a classe positiva representa inadimplência e o dataset apresenta desbalanceamento moderado.

### Reprodutibilidade
O pipeline completo é persistido em `models/credit_scoring_baseline.joblib`, garantindo que imputação, encoding, scaling e modelo sejam reutilizados como uma única unidade.

### Próximas etapas
1. revisar as métricas obtidas;
2. modularizar treinamento e avaliação em `train.py` e `evaluate.py`;
3. criar o Data Contract com Pandera;
4. construir um lote inválido e demonstrar seu bloqueio antes do modelo.

---
### Nota antes do commit

Execute **Run All** com o kernel `.venv`.

O notebook está pronto quando:

- o treinamento termina sem `MemoryError`;
- as métricas são exibidas;
- os gráficos são gerados;
- `credit_scoring_baseline.joblib` é salvo;
- o teste final informa `Pipeline salvo e recarregado com sucesso.`

Se uma célula falhar, corrija antes de executar as posteriores.